# 處方資料分析

本 Notebook 載入 `batch_process.py` 產生的 CSV，進行探索性資料分析 (EDA)。

**執行前準備：**
1. 把處方圖片放入 `../data/input/`
2. 執行 `python batch_process.py` 產生 CSV
3. 修改下方 `CSV_PATH` 指向產生的檔案

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path

matplotlib.rc('font', family='Microsoft JhengHei')  # Windows 繁中字體
plt.rcParams['axes.unicode_minus'] = False

# ── 載入最新的 CSV ────────────────────────────────────────────────────────────
output_dir = Path('../data/output')
csvs = sorted(output_dir.glob('prescriptions_*.csv'), reverse=True)
CSV_PATH = csvs[0] if csvs else None

if CSV_PATH is None:
    raise FileNotFoundError('找不到 CSV，請先執行 batch_process.py')

df = pd.read_csv(CSV_PATH, encoding='utf-8-sig',
                 parse_dates=['prescription_date'])
print(f'載入：{CSV_PATH.name}')
print(f'共 {len(df)} 筆藥品記錄，{df["source_file"].nunique()} 張處方')
df.head()

## 基本統計

In [ ]:
print('=== 欄位完整度 ===')
print(df.isnull().mean().mul(100).round(1).astype(str) + '%')
print()
print('=== 數值欄位統計 ===')
df[['duration_days']].describe().round(1)

## 最常見藥品 Top 20

In [ ]:
top_meds = df['med_name'].dropna().value_counts().head(20)

fig, ax = plt.subplots(figsize=(8, 6))
top_meds.sort_values().plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('處方次數')
ax.set_title('最常見藥品 Top 20')
plt.tight_layout()
plt.show()
top_meds

## 診斷分布

In [ ]:
per_rx = df.drop_duplicates('source_file')
diag_counts = per_rx['diagnosis'].dropna().value_counts().head(15)

fig, ax = plt.subplots(figsize=(6, 6))
diag_counts.plot.pie(ax=ax, autopct='%1.1f%%', startangle=90)
ax.set_ylabel('')
ax.set_title('診斷分布 Top 15')
plt.tight_layout()
plt.show()

## 每月處方量趨勢

In [ ]:
monthly = (
    per_rx.dropna(subset=['prescription_date'])
    .set_index('prescription_date')
    .resample('ME')['source_file']
    .count()
)

fig, ax = plt.subplots(figsize=(10, 4))
monthly.plot(ax=ax, marker='o', color='steelblue')
ax.set_xlabel('月份')
ax.set_ylabel('處方數')
ax.set_title('每月處方量趨勢')
plt.tight_layout()
plt.show()

## 給藥頻次分析

In [ ]:
freq_counts = df['frequency'].dropna().value_counts().head(10)

fig, ax = plt.subplots(figsize=(8, 4))
freq_counts.plot.bar(ax=ax, color='seagreen', rot=30)
ax.set_ylabel('次數')
ax.set_title('給藥頻次分布')
plt.tight_layout()
plt.show()

## 匯出摘要報告

In [ ]:
summary = {
    '總處方數': df['source_file'].nunique(),
    '總藥品記錄數': len(df),
    '不重複藥品種類': df['med_name'].nunique(),
    '涉及醫院數': df['hospital'].nunique(),
    '涉及醫師數': df['doctor_name'].nunique(),
    '最常見藥品': top_meds.index[0] if len(top_meds) else None,
    '平均療程天數': df['duration_days'].mean().round(1),
}
pd.Series(summary, name='值').to_frame()